In [1]:
#import user modules
#--- MATPLOTLIB
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.markers import MarkerStyle
from matplotlib.gridspec import GridSpec
import matplotlib.colors as mcolors
import seaborn as sns
import pandas as pd
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

import sys
my_path = "../../Python/"
if my_path not in sys.path:
    sys.path.append(my_path)
 
for place in sys.path: 
    print(place)

from tools import *
from fit_funcs import *
from entropy import *
import costfun.costfun as cost
import utils.figures as fig_help
from RandomMatrixTheory import goe
import utils.tools as tools

import importlib as imp
def reload_modules():
    imp.reload(cost)
    imp.reload(fig_help)

import itertools
matplotlib.rcParams['mathtext.fontset'] = 'cm'
# plt.rcParams["mathtext.fontset"] = "cm"
matplotlib.rcParams['font.family'] = 'STIXGeneral'
latex_engine = 'xelatex'
latex_elements = {'preamble':r'\usepackage{physics}'}

%matplotlib inline
colors_ls = (list(mcolors.TABLEAU_COLORS)[:200])
colors_ls_cyc = itertools.cycle(colors_ls)

markers_ls = ['o','s','v', 'D', '<', 'X', '^', '*', '+']
markers = itertools.cycle(markers_ls)

linestyle_ls = ['-','--',':', '-.']
linestyle = itertools.cycle(linestyle_ls)

#--- NUMERICAL LIBS
import numpy as np
import itertools
import math
import random
from cmath import nan
import h5py   
from scipy import stats

# SCIPY LIBS
import scipy.stats as statistics
from scipy.special import binom
from scipy.special import erfinv
from scipy.special import erf
from scipy.special import digamma
from scipy.special import polygamma
from scipy.special import lambertw
from scipy.optimize import curve_fit as fit
from scipy.signal import savgol_filter
from scipy import integrate
from scipy import fft
from scipy.interpolate import UnivariateSpline as InterpolateSpline
from scipy.interpolate import splrep, splev
from scipy.interpolate import make_interp_spline as make_spline
from scipy.interpolate import interp1d
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import fsolve

from mpl_toolkits.axes_grid1 import make_axes_locatable

# OTHER
import warnings
warnings.filterwarnings('ignore')
from joblib import Parallel, delayed
import copy
import os
from os import sep as kPSep
from os.path import exists


config_disorder = 0
scaled_disorder = 0

if config_disorder:
    base_dir = "../results_conf_dis/"
else:
    base_dir = "../results/"

print(base_dir[2:])
%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}


/Users/rafal.swietek/Projects/CODES/QHamSolver/QuantumSun/Jupyter_Python
/Users/rafal.swietek/opt/anaconda3/lib/python39.zip
/Users/rafal.swietek/opt/anaconda3/lib/python3.9
/Users/rafal.swietek/opt/anaconda3/lib/python3.9/lib-dynload

/Users/rafal.swietek/opt/anaconda3/lib/python3.9/site-packages
/Users/rafal.swietek/opt/anaconda3/lib/python3.9/site-packages/aeosa
/Users/rafal.swietek/opt/anaconda3/lib/python3.9/site-packages/locket-0.2.1-py3.9.egg
/Users/rafal.swietek/opt/anaconda3/lib/python3.9/site-packages/IPython/extensions
/Users/rafal.swietek/.ipython
../../Python/
/results/


In [11]:
L=8
J=1.0
alfa=1.0
h=1.0
w=0.5
zeta=0.2
gamma=1.0

conf_disorder = False

N=3
J=1
scaled_dis = 0

ini_ave=1
num_realis = 30000

def loop_body(L, alfa):
    L_total = L+N
    dim = 2**L_total

    S_GS = np.zeros((L_total + 1))
    S_EX = np.zeros((L_total + 1))
    S_site_GS = np.zeros((L_total + 1))
    S_site_EX = np.zeros((L_total + 1))
    
    Sx_GS = np.zeros((L_total))
    Sy_GS = np.zeros((L_total), dtype=complex)
    Sz_GS = np.zeros((L_total))
    Sx_EX = np.zeros((L_total))
    Sy_EX = np.zeros((L_total), dtype=complex)
    Sz_EX = np.zeros((L_total))
    
    SxSx_GS = np.zeros((L_total, L_total))
    SySy_GS = np.zeros((L_total, L_total))
    SzSz_GS = np.zeros((L_total, L_total))
    SxSx_EX = np.zeros((L_total, L_total))
    SySy_EX = np.zeros((L_total, L_total))
    SzSz_EX = np.zeros((L_total, L_total))
    counter = 0

    # w = np.round(w, 6)
    alfa = np.round(alfa, 6)

    print(info(L=L, N=N, J=J, gamma=gamma, zeta=zeta, alfa=alfa, h=h, w=w, ext='.hdf5', scaled_disorder=scaled_dis, ini_ave = ini_ave), flush=True)

    energy_gap = 0
    for real in range(num_realis):
        name = base_dir + 'GroundState/TESTS/realisation=%d/'%real + info(L=L, N=N, J=J, gamma=gamma, zeta=zeta, alfa=alfa, h=h, w=w, ini_ave = ini_ave, ext='.hdf5', scaled_disorder=scaled_dis)
        if exists(name):
            with h5py.File(name, "r") as file:
                try:
                    _var_names1 = ['energies', 'entropy GS', 'single_site_entropy GS', 'entropy excited', 'single_site_entropy excited', 'subsystem sizes']
                    _var_names2 = ['Sx GS', 'Sy GS', 'Sz GS', 'Sx excited', 'Sy excited', 'Sz excited', 'SxSx GS', 'SySy GS', 'SzSz GS', 'SxSx excited', 'SySy excited', 'SzSz excited']
                    for _name_var in [*_var_names1, *_var_names2]:
                        _name_var_funt = _name_var;
                        xx = np.array(file.get(_name_var)[0])
                    
                except (IndexError, OSError, TypeError) as e:
                    print("FAILED AT: ", _name_var_funt, "\tCorrupted file!", name, flush=True)
                    print("\t\tERROR", e, flush=True)
                
                E = np.array(file.get('energies')[0])
                energy_gap += E[1] - E[0]
                
                # ENTANGLEMENT ENTROPIES
                S_GS += np.array(file.get('entropy GS')[0])
                S_EX += np.array(file.get('entropy excited')[0])
                S_site_GS += np.array(file.get('single_site_entropy GS')[0])
                S_site_EX += np.array(file.get('single_site_entropy excited')[0])
                
                # EXPECTATION VALUES
                Sx_GS += np.array(file.get('Sx GS')[0])
                Sy_GS += np.array(file.get('Sy GS')[0]).view(complex)
                Sz_GS += np.array(file.get('Sz GS')[0])
                Sx_EX += np.array(file.get('Sx excited')[0])
                Sy_EX += np.array(file.get('Sy excited')[0]).view(complex)
                Sz_EX += np.array(file.get('Sz excited')[0])
                
                # CORRELATION MATRICES
                SxSx_GS += np.array(file.get('SxSx GS'))
                SySy_GS += np.array(file.get('SySy GS'))
                SzSz_GS += np.array(file.get('SzSz GS'))
                SxSx_EX += np.array(file.get('SxSx excited'))
                SySy_EX += np.array(file.get('SySy excited'))
                SzSz_EX += np.array(file.get('SzSz excited'))
                
                counter += 1
        else:
            asdbcinoas=1
#                print(real)
    print(L, gamma, alfa, counter, flush=True)
    if counter > 0:
        name = base_dir + 'GroundState/TESTS/' + info(L=L, N=N, J=J, gamma=gamma, zeta=zeta, alfa=alfa, h=h, w=w, ext='.hdf5', scaled_disorder=scaled_dis)

        hf = h5py.File(name, 'w')
        hf['energy gap'] = energy_gap / counter
        hf['realisations'] = counter
        
        hf.create_dataset('entropies GS', S_GS.shape, data = S_GS / counter)
        hf.create_dataset('single_site_entropy GS', S_site_GS.shape, data = S_site_GS / counter)
        hf.create_dataset('entropies excited', S_EX.shape, data = S_EX / counter)
        hf.create_dataset('single_site_entropy excited', S_site_EX.shape, data = S_site_EX / counter)
        
        hf.create_dataset('Sx GS', Sx_GS.shape, data = Sx_GS / counter)
        hf.create_dataset('Sy GS', Sy_GS.shape, data = Sy_GS / counter)
        hf.create_dataset('Sz GS', Sz_GS.shape, data = Sz_GS / counter)
        
        hf.create_dataset('Sx excited', Sx_EX.shape, data = Sx_EX / counter)
        hf.create_dataset('Sy excited', Sy_EX.shape, data = Sy_EX / counter)
        hf.create_dataset('Sz excited', Sz_EX.shape, data = Sz_EX / counter)
        
        hf.create_dataset('SxSx GS', SxSx_GS.shape, data = SxSx_GS / counter)
        hf.create_dataset('SySy GS', SySy_GS.shape, data = SySy_GS / counter)
        hf.create_dataset('SzSz GS', SzSz_GS.shape, data = SzSz_GS / counter)
        
        hf.create_dataset('SxSx excited', SxSx_EX.shape, data = SxSx_EX / counter)
        hf.create_dataset('SySy excited', SySy_EX.shape, data = SySy_EX / counter)
        hf.create_dataset('SzSz excited', SzSz_EX.shape, data = SzSz_EX / counter)
        
        


        hf.close()


L = 13
alfa = 0.8
loop_body(L, alfa)

_L=13,N=3,J=1,g=1,zeta=0.2,alfa=0.8,h=1,w=0.5,ini_ave.hdf5
13 1.0 0.8 3
